In [0]:
%sql
CREATE TABLE IF NOT EXISTS finance.liquidity.assumption_reference (
framework STRING COMMENT 'Y15, LCR, ODM',
assumption_category STRING COMMENT 'Deposit, Product, Behavioral, Regulatory, Classification',
assumption_code STRING COMMENT 'Stable business key, e.g. LCR_RUNOFF',
assumption_name STRING COMMENT 'Business name of assumption',
assumption_value STRING COMMENT 'Value assigned to assumption',
assumption_description STRING COMMENT 'Explanation of assumption',
effective_date DATE,
expiration_date DATE,
active_flag STRING COMMENT 'Y/N',
source_system STRING,
owner STRING,
comments STRING,
created_by STRING,
created_ts TIMESTAMP,
modified_by STRING,
modified_ts TIMESTAMP
)
USING DELTA;
DESCRIBE finance.liquidity.assumption_reference;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS finance.liquidity.assumption_reference_bkp_lcr_20261102 AS
SELECT * FROM finance.liquidity.assumption_reference WHERE framework = 'LCR';
CREATE OR REPLACE TABLE finance.liquidity.assumption_reference_stg_lcr AS
SELECT
framework, assumption_category, assumption_code, assumption_name, assumption_value,
assumption_description, effective_date, expiration_date, active_flag, source_system,
owner, comments, created_by, created_ts, modified_by, modified_ts
FROM finance.liquidity.assumption_reference
WHERE framework = 'LCR'
QUALIFY ROW_NUMBER() OVER (
PARTITION BY assumption_code, effective_date
ORDER BY modified_ts DESC, created_ts DESC
) = 1;
DELETE FROM finance.liquidity.assumption_reference WHERE framework = 'LCR';
INSERT INTO finance.liquidity.assumption_reference (
framework, assumption_category, assumption_code, assumption_name, assumption_value,
assumption_description, effective_date, expiration_date, active_flag, source_system,
owner, comments, created_by, created_ts, modified_by, modified_ts
)
SELECT * FROM finance.liquidity.assumption_reference_stg_lcr;
DROP TABLE IF EXISTS finance.liquidity.assumption_reference_stg_lcr;
SELECT assumption_code, effective_date, COUNT(1) as row_count
FROM finance.liquidity.assumption_reference
WHERE framework = 'LCR'
GROUP BY assumption_code, effective_date
ORDER BY assumption_code;


In [0]:
%sql
-- Part A: Seed LCR assumptions (MERGE - existing rows are skipped)
-- Part B: LCR runoff factors table (rebuilt each run, history kept by Delta)
-- To CHANGE an existing assumption value, use Cell 4 instead.

-- Part A: add new LCR assumptions as rows in the VALUES list
MERGE INTO finance.liquidity.assumption_reference AS tgt
USING (
SELECT * FROM VALUES
('LCR', 'Behavioral', 'LCR_RUNOFF', 'Retail Runoff Rate', '7%',
'LCR Retail Deposit Runoff Rate', DATE'2026-10-28', 'Treasury', 'Treasury')
AS s(framework, assumption_category, assumption_code, assumption_name, assumption_value,
assumption_description, effective_date, source_system, owner)
) AS src
ON tgt.framework = src.framework
AND tgt.assumption_code = src.assumption_code
AND tgt.effective_date = src.effective_date
WHEN NOT MATCHED THEN INSERT (
framework, assumption_category, assumption_code, assumption_name, assumption_value,
assumption_description, effective_date, expiration_date, active_flag, source_system,
owner, comments, created_by, created_ts, modified_by, modified_ts
)
VALUES (
src.framework, src.assumption_category, src.assumption_code, src.assumption_name, src.assumption_value,
src.assumption_description, src.effective_date, NULL, 'Y', src.source_system,
src.owner, NULL, current_user(), current_timestamp(), current_user(), current_timestamp()
);
-- Part B: runoff factors by lcr_class (join-safe key, no spaces) -- 2026-11-12
-- One row per lcr_class (45) + 3 excess operational rows (joined on rollup via lcr_rollup_2).
-- 2026-11-13 changes: BDSPLT30 0.10 added; secured split into NMD / TimeLT30 / TimeGT30
--   (TimeLT30 1.00, NMD and TimeGT30 0.20).
-- rollup = descriptive name; it is NOT unique (several lcr_class rows share a rollup).
-- 2026-11-12 changes: Wholesale Traditional non-op NMD Insured 0.20 / Uninsured 0.40 (were missing);
--   Wholesale Reciprocal TD GT30 0.40 -> 0.20; secured split into SecuredPSE / SecuredNonPSE.
CREATE OR REPLACE TABLE finance.liquidity.lcr_runoff_factors
COMMENT 'LCR runoff factor by lcr_class (join key). rollup is the descriptive name.'
AS
SELECT
lcr_class,
rollup,
CAST(runoff_factor AS DECIMAL(5,4)) AS runoff_factor,
current_user() AS loaded_by,
current_timestamp() AS loaded_ts
FROM VALUES
-- BDSP
('BDSPLT30', 'BDSP Sweep Deposits', 0.10),
-- Brokered
('BrokeredNMDLT30', 'Brokered NMDs LT30', 0.40),
('BrokeredNMDGT30', 'Brokered NMDs GT30', 0.25),
('BrokeredTimeLT30', 'Brokered CDs LT30', 0.40),
('BrokeredTimeGT30', 'Brokered CDs GT30', 0.25),
-- Operational
('NFSEOperationalInsured', 'NFSE Insured Operational Deposits', 0.05),
('NFSEOperationalUninsured', 'NFSE Uninsured Operational Deposits', 0.25),
('FSEOperationalInsured', 'FSE Insured Operational Deposits', 0.05),
('FSEOperationalUninsured', 'FSE Uninsured Operational Deposits', 0.25),
-- FSE catch-all (100%)
('FSEReciprocalNMD', 'FSE Reciprocal Wholesale NMD', 1.00),
('FSEReciprocalTimeLT30', 'FSE Reciprocal Wholesale TD LT30', 1.00),
('FSEReciprocalTimeGT30', 'FSE Reciprocal Wholesale TD GT30', 1.00),
('FSETraditionalNMD', 'FSE Non-Operational NMD', 1.00),
('FSETraditionalTimeLT30', 'Other FSE Deposits', 1.00),
('FSETraditionalTimeGT30', 'Other FSE Deposits', 1.00),
-- Wholesale (NFSE, non-operational)
('WholesaleTraditionalInsuredNMD', 'NFSE Insured Non-Operational NMD', 0.20),
('WholesaleTraditionalUninsuredNMD', 'NFSE Uninsured Non-Operational NMD', 0.40),
('WholesaleTraditionalInsuredTimeLT30', 'NFSE Insured Traditional Wholesale TD LT30', 0.20),
('WholesaleTraditionalInsuredTimeGT30', 'NFSE Insured Traditional Wholesale TD GT30', 0.10),
('WholesaleTraditionalUninsuredTimeLT30', 'NFSE Uninsured Traditional Wholesale TD LT30', 0.40),
('WholesaleTraditionalUninsuredTimeGT30', 'NFSE Uninsured Traditional Wholesale TD GT30', 0.25),
('WholesaleReciprocalInsuredNMD', 'NFSE Reciprocal Wholesale NMD', 0.40),
('WholesaleReciprocalUninsuredNMD', 'NFSE Reciprocal Wholesale NMD', 0.40),
('WholesaleReciprocalInsuredTimeLT30', 'NFSE Reciprocal Wholesale TD LT30', 1.00),
('WholesaleReciprocalUninsuredTimeLT30', 'NFSE Reciprocal Wholesale TD LT30', 1.00),
('WholesaleReciprocalInsuredTimeGT30', 'NFSE Reciprocal Wholesale TD GT30', 0.20),
('WholesaleReciprocalUninsuredTimeGT30', 'NFSE Reciprocal Wholesale TD GT30', 0.20),
-- Retail
('RetailTraditionalInsuredNMD', 'Stable Retail NMDs', 0.03),
('RetailTraditionalInsuredTimeLT30', 'Stable Retail TDs LT30', 0.03),
('RetailTraditionalInsuredTimeGT30', 'Stable Retail TDs GT30', 0.03),
('RetailTraditionalUninsuredNMD', 'Other Retail NMDs', 0.10),
('RetailTraditionalUninsuredTimeLT30', 'Other Retail TDs LT30', 0.10),
('RetailTraditionalUninsuredTimeGT30', 'Other Retail TDs GT30', 0.10),
('RetailReciprocalInsuredNMD', 'Insured Reciprocal Retail NMDs', 0.05),
('RetailReciprocalInsuredTimeLT30', 'Insured Reciprocal Retail TDs LT30', 0.05),
('RetailReciprocalInsuredTimeGT30', 'Insured Reciprocal Retail TDs GT30', 0.05),
('RetailReciprocalUninsuredNMD', 'Uninsured Reciprocal Retail NMDs', 0.10),
('RetailReciprocalUninsuredTimeLT30', 'Uninsured Reciprocal Retail TDs LT30', 0.10),
('RetailReciprocalUninsuredTimeGT30', 'Uninsured Reciprocal Retail TDs GT30', 0.10),
-- Secured (last)
('SecuredPSENMD', 'PSE Secured Deposit', 0.20),
('SecuredPSETimeLT30', 'PSE Secured Deposit', 1.00),
('SecuredPSETimeGT30', 'PSE Secured Deposit', 0.20),
('SecuredNonPSENMD', 'Non-PSE Secured Deposit', 0.20),
('SecuredNonPSETimeLT30', 'Non-PSE Secured Deposit', 1.00),
('SecuredNonPSETimeGT30', 'Non-PSE Secured Deposit', 0.20),
-- Secured reciprocal (2026-11-17, per user: 40%, no maturity split)
('SecuredRecipDepositsPSE', 'PSE Secured Reciprocal Deposit', 0.40),
('SecuredRecipDepositsNonPSE', 'Non-PSE Secured Reciprocal Deposit', 0.40),
-- Excess operational (second balance; pipeline joins these on rollup = lcr_rollup_2)
('NFSEExcessOperationalInsured', 'NFSE Insured Excess Operational Deposits', 0.25),
('NFSEExcessOperationalUninsured', 'NFSE Uninsured Excess Operational Deposits', 0.40),
('FSEExcessOperational', 'FSE Excess Operational Deposits', 1.00)
AS t(lcr_class, rollup, runoff_factor);
SELECT * FROM finance.liquidity.lcr_runoff_factors ORDER BY lcr_class;


In [0]:
%sql
-- Expires the current row (active_flag = 'N') and inserts a new active row.
-- If the value is unchanged, nothing happens. Safe to re-run


CREATE OR REPLACE TEMP VIEW lcr_assumption_change AS
SELECT * FROM VALUES
('LCR_RUNOFF', '7%', DATE'2026-11-02', 'Describe reason for change')
AS c(assumption_code, new_value, new_effective_date, change_comment);
-- Step 1: expire the current active row
MERGE INTO finance.liquidity.assumption_reference AS tgt
USING lcr_assumption_change AS c
ON tgt.framework = 'LCR'
AND tgt.assumption_code = c.assumption_code
AND tgt.active_flag = 'Y'
WHEN MATCHED AND tgt.assumption_value <> c.new_value THEN UPDATE SET
tgt.active_flag = 'N',
tgt.expiration_date = date_sub(c.new_effective_date, 1),
tgt.modified_by = current_user(),
tgt.modified_ts = current_timestamp();
-- Step 2: insert the new active row (copies other attributes from the latest row)
INSERT INTO finance.liquidity.assumption_reference (
framework, assumption_category, assumption_code, assumption_name, assumption_value,
assumption_description, effective_date, expiration_date, active_flag, source_system,
owner, comments, created_by, created_ts, modified_by, modified_ts
)
SELECT
p.framework, p.assumption_category, p.assumption_code, p.assumption_name, c.new_value,
p.assumption_description, c.new_effective_date, CAST(NULL AS DATE), 'Y', p.source_system,
p.owner, c.change_comment, current_user(), current_timestamp(), current_user(), current_timestamp()
FROM lcr_assumption_change AS c
JOIN (
SELECT *
FROM finance.liquidity.assumption_reference
WHERE framework = 'LCR'
QUALIFY ROW_NUMBER() OVER (
PARTITION BY assumption_code
ORDER BY effective_date DESC, modified_ts DESC
) = 1
) AS p
ON p.assumption_code = c.assumption_code
WHERE NOT EXISTS (
SELECT 1
FROM finance.liquidity.assumption_reference x
WHERE x.framework = 'LCR'
AND x.assumption_code = c.assumption_code
AND x.active_flag = 'Y'
);
SELECT assumption_code, assumption_value, effective_date, expiration_date, active_flag, comments
FROM finance.liquidity.assumption_reference
WHERE framework = 'LCR'
AND assumption_code IN (SELECT assumption_code FROM lcr_assumption_change)
ORDER BY effective_date DESC;


In [0]:
%sql
WITH lcr AS (
SELECT * FROM finance.liquidity.assumption_reference WHERE framework = 'LCR'
),
rf AS (
SELECT * FROM finance.liquidity.lcr_runoff_factors
),
checks AS (
SELECT 1 AS check_order, 'Duplicate active assumption rows' AS check_name, COUNT(1) AS issues
FROM (SELECT assumption_code FROM lcr WHERE active_flag = 'Y'
GROUP BY assumption_code HAVING COUNT(1) > 1)
UNION ALL
SELECT 2, 'Assumption codes with no active row', COUNT(1)
FROM (SELECT assumption_code FROM lcr GROUP BY assumption_code
HAVING SUM(CASE WHEN active_flag = 'Y' THEN 1 ELSE 0 END) = 0)
UNION ALL
SELECT 3, 'Runoff factor count is not 50', CASE WHEN COUNT(1) = 50 THEN 0 ELSE 1 END FROM rf
UNION ALL
SELECT 4, 'Duplicate runoff lcr_class', COUNT(1) - COUNT(DISTINCT lcr_class) FROM rf
UNION ALL
SELECT 5, 'Runoff factors outside 0-1', COUNT_IF(runoff_factor < 0 OR runoff_factor > 1) FROM rf
)
SELECT
check_name,
issues,
CASE WHEN issues = 0 THEN 'PASS' ELSE 'FAIL' END AS status
FROM checks
ORDER BY check_order;


In [0]:
%sql
-- LCR report lines and the lcr_class -> line map (moved from liquidity/LCR/Create/LCR_Import_Package, 2026-11-17).
-- Both tables are rebuilt each run; history is kept by Delta (DESCRIBE HISTORY).
-- The DepositsPlus pipeline reads them in 00_runoff_factors.sql and stamps the line on each
-- account in deposits_reg_class. To change a mapping, edit the VALUES here and rerun this cell.
-- rule SPLIT : op_balance -> line_number_op, ledger_balance - op_balance -> line_number
-- rule LEDGER: all of the balance -> line_number

CREATE OR REPLACE TABLE finance.liquidity.lcr_line_def
COMMENT 'LCR report line definitions (line_number key). Maintained in the LCR Assumptions notebook.'
AS
SELECT *, current_user() AS loaded_by, current_timestamp() AS loaded_ts
FROM VALUES
    ('1.01', '1', 'Retail Deposits', 'Stable Retail - Insured (NMD and TD >30d)', '32.a.1', 'OUTFLOW'),
    ('1.02', '1', 'Retail Deposits', 'Stable Retail - Insured TDs <=30d', '32.a.1.LT30', 'OUTFLOW'),
    ('1.03', '1', 'Retail Deposits', 'Other Retail - Uninsured (NMD and TD >30d)', '32.a.2', 'OUTFLOW'),
    ('1.04', '1', 'Retail Deposits', 'Other Retail - Uninsured TDs <=30d', '32.a.2.LT30', 'OUTFLOW'),
    ('2.01', '2', 'Brokered, Reciprocal & Sweeps (Retail)', 'Consumer ICS - Insured (NMD and >30d)', '32.g.5', 'OUTFLOW'),
    ('2.02', '2', 'Brokered, Reciprocal & Sweeps (Retail)', 'CDARS maturing <=30d - Insured', '32.g.5.LT30', 'OUTFLOW'),
    ('2.03', '2', 'Brokered, Reciprocal & Sweeps (Retail)', 'Consumer ICS - Uninsured (NMD and >30d)', '32.g.6', 'OUTFLOW'),
    ('2.04', '2', 'Brokered, Reciprocal & Sweeps (Retail)', 'Consumer ICS - Uninsured <=30d', '32.g.6.LT30', 'OUTFLOW'),
    ('2.05', '2', 'Brokered, Reciprocal & Sweeps (Retail)', 'Insured Affiliated Sweeps (BDSP)', '32.g.7', 'OUTFLOW'),
    ('2.06', '2', 'Brokered, Reciprocal & Sweeps (Retail)', 'Brokered CDs maturing <=30d', '32.g.1', 'OUTFLOW'),
    ('2.07', '2', 'Brokered, Reciprocal & Sweeps (Retail)', 'Brokered CDs maturing >30d', '32.g.2', 'OUTFLOW'),
    ('2.08', '2', 'Brokered, Reciprocal & Sweeps (Retail)', 'Brokered NMD', '32.g.3', 'OUTFLOW'),
    ('2.09', '2', 'Brokered, Reciprocal & Sweeps (Retail)', 'Brokered Term NMD', '32.g.3.Term', 'OUTFLOW'),
    ('3.01', '3', 'Unsecured Wholesale', 'Operational Commercial - Insured', '32.h.4 - Insured', 'OUTFLOW'),
    ('3.02', '3', 'Unsecured Wholesale', 'Operational Commercial - Uninsured', '32.h.4 - Uninsured', 'OUTFLOW'),
    ('3.03', '3', 'Unsecured Wholesale', 'Non-Operational Commercial - Insured (incl. excess operational)', '32.h.1.i', 'OUTFLOW'),
    ('3.04', '3', 'Unsecured Wholesale', 'Non-Operational Commercial - Uninsured (incl. excess operational)', '32.h.1.ii.A', 'OUTFLOW'),
    ('3.05', '3', 'Unsecured Wholesale', 'Wholesale TDs maturing <=30d', '32.h.1', 'OUTFLOW'),
    ('3.06', '3', 'Unsecured Wholesale', 'Insured TDs maturing >30d', '32.h.1.Insured', 'OUTFLOW'),
    ('3.07', '3', 'Unsecured Wholesale', 'Uninsured TDs maturing >30d', '32.h.1.Uninsured', 'OUTFLOW'),
    ('3.08', '3', 'Unsecured Wholesale', 'Commercial ICS - Reciprocal NMDs', '32.h.1.ii.B', 'OUTFLOW'),
    ('3.09', '3', 'Unsecured Wholesale', 'Commercial ICS - Reciprocal TDs <=30d', '32.h.1.ii.B', 'OUTFLOW'),
    ('3.10', '3', 'Unsecured Wholesale', 'Commercial ICS - Reciprocal TDs >30d', '32.h.1.ii.B', 'OUTFLOW'),
    ('3.11', '3', 'Unsecured Wholesale', 'Insured Sweep Wholesale', '32.h.1.ii.B', 'OUTFLOW'),
    ('3.12', '3', 'Unsecured Wholesale', 'Uninsured Sweep Wholesale', '32.h.1.ii.B', 'OUTFLOW'),
    ('3.13', '3', 'Unsecured Wholesale', 'Unsecured Portion of Secured Funding', '32.h', 'OUTFLOW'),
    ('3.14', '3', 'Unsecured Wholesale', 'Financial Sector Entity - Non-Operational (incl. FSE excess operational)', '32.h.2', 'OUTFLOW'),
    ('4.01', '4', 'Secured Funding', 'Secured - Level 1 collateral', '32.j.1.i', 'OUTFLOW'),
    ('4.02', '4', 'Secured Funding', 'Secured - Level 2A collateral', '32.j.1.ii', 'OUTFLOW'),
    ('4.03', '4', 'Secured Funding', 'Secured - Level 2B collateral', '32.j.1.iv', 'OUTFLOW'),
    ('4.04', '4', 'Secured Funding', 'Secured - Non-HQLA collateral', '32.j.1.vi', 'OUTFLOW'),
    ('4.05', '4', 'Secured Funding', 'FHLB Advances', 'TBD', 'OUTFLOW'),
    ('4.06', '4', 'Secured Funding', 'FHLB Advances maturing >30d (memo)', 'n/a', 'MEMO'),
    ('5.01', '5', 'Contractual Obligations & Collateral', 'Fed Funds Purchased', 'TBD', 'OUTFLOW'),
    ('5.02', '5', 'Contractual Obligations & Collateral', 'Debt Maturing <=30 Days (LTD)', 'TBD', 'OUTFLOW'),
    ('5.03', '5', 'Contractual Obligations & Collateral', 'Collateral Outflow', '32.f', 'OUTFLOW'),
    ('5.04', '5', 'Contractual Obligations & Collateral', 'Long-Term Debt maturing >30d (memo)', 'n/a', 'MEMO'),
    ('5.06', '5', 'Contractual Obligations & Collateral', 'Fed Funds Purchased maturing >30d (memo)', 'n/a', 'MEMO'),
    ('6.01', '6', 'Unfunded Commitments', 'Mortgage Commitments', '32.d', 'OUTFLOW'),
    ('6.02', '6', 'Unfunded Commitments', 'Retail Loan Commitments', '32.e.ii', 'OUTFLOW'),
    ('6.03', '6', 'Unfunded Commitments', 'Credit Facility - Commercial (Working Capital/Other)', '32.e.iii', 'OUTFLOW'),
    ('6.04', '6', 'Unfunded Commitments', 'Liquidity Facility - Commercial Customer', '32.e.iv', 'OUTFLOW'),
    ('6.05', '6', 'Unfunded Commitments', 'Credit or Liquidity Facility - Bank', '32.e.v', 'OUTFLOW'),
    ('6.06', '6', 'Unfunded Commitments', 'Credit Facility - Non-Bank Financial', '32.e.vi', 'OUTFLOW'),
    ('6.07', '6', 'Unfunded Commitments', 'Liquidity Facility - Non-Bank Financial', '32.e.vii', 'OUTFLOW'),
    ('6.08', '6', 'Unfunded Commitments', 'Liquidity Facility - SPEs', '32.e.viii', 'OUTFLOW'),
    ('7.01', '7', 'Inflows', 'Retail Loan Inflows', 'TBD', 'INFLOW'),
    ('7.02', '7', 'Inflows', 'Commercial Loan Inflows', 'TBD', 'INFLOW'),
    ('8.01', '8', 'Secured Deposit Balances', 'Secured deposits - PSE (balance, see Secured)', 'n/a', 'MEMO'),
    ('8.02', '8', 'Secured Deposit Balances', 'Secured deposits - Non-PSE (balance, see Secured)', 'n/a', 'MEMO'),
    ('9.99', '9', 'Review', 'Unclassified wholesale funding (review)', 'n/a', 'REVIEW')
  AS t(line_number, section_number, section_name, line_name, schedule_ref, line_type);

-- Old codes SecuredPSE / SecuredNonPSE dropped (reg_class splits secured into NMD / TimeLT30 / TimeGT30).
CREATE OR REPLACE TABLE finance.liquidity.lcr_class_map
COMMENT 'LCR class to report line (lcr_line_def). Key: source_system + lcr_class. Maintained in the LCR Assumptions notebook.'
AS
SELECT *, current_user() AS loaded_by, current_timestamp() AS loaded_ts
FROM VALUES
    ('deposits_reg', 'BDSPLT30', 'LEDGER', NULL, '2.05', 'Brokered', 'Insured', 'BDSP sweep class. Maps to line 2.05 (32.g.7). Confirm the line.'),
    ('deposits_reg', 'BrokeredNMDLT30', 'LEDGER', NULL, '2.08', 'Brokered', 'n/a', 'Extract has no insured flag; all treated as 32.g.3 (insured)'),
    ('deposits_reg', 'BrokeredTimeGT30', 'LEDGER', NULL, '2.07', 'Brokered', 'n/a', NULL),
    ('deposits_reg', 'BrokeredTimeLT30', 'LEDGER', NULL, '2.06', 'Brokered', 'n/a', NULL),
    ('deposits_reg', 'FSEOperationalInsured', 'SPLIT', '3.01', '3.14', 'FSE', 'Insured', 'op_balance to 3.01; remainder to FSE non-op 32.h.2. Confirm.'),
    ('deposits_reg', 'FSEOperationalUninsured', 'SPLIT', '3.02', '3.14', 'FSE', 'Uninsured', 'op_balance to 3.02; remainder to FSE non-op 32.h.2. Confirm.'),
    ('deposits_reg', 'FSEReciprocalNMD', 'LEDGER', NULL, '3.14', 'FSE', 'n/a', NULL),
    ('deposits_reg', 'FSEReciprocalTimeGT30', 'LEDGER', NULL, '3.14', 'FSE', 'n/a', NULL),
    ('deposits_reg', 'FSEReciprocalTimeLT30', 'LEDGER', NULL, '3.14', 'FSE', 'n/a', NULL),
    ('deposits_reg', 'FSETraditionalNMD', 'LEDGER', NULL, '3.14', 'FSE', 'n/a', NULL),
    ('deposits_reg', 'FSETraditionalTimeGT30', 'LEDGER', NULL, '3.14', 'FSE', 'n/a', NULL),
    ('deposits_reg', 'FSETraditionalTimeLT30', 'LEDGER', NULL, '3.14', 'FSE', 'n/a', NULL),
    ('deposits_reg', 'NFSEOperationalInsured', 'SPLIT', '3.01', '3.03', 'NFSE', 'Insured', 'op_balance to 3.01; remainder (excess) to 3.03'),
    ('deposits_reg', 'NFSEOperationalUninsured', 'SPLIT', '3.02', '3.04', 'NFSE', 'Uninsured', 'op_balance to 3.02; remainder (excess) to 3.04'),
    ('deposits_reg', 'RetailReciprocalInsuredNMD', 'LEDGER', NULL, '2.01', 'Retail', 'Insured', NULL),
    ('deposits_reg', 'RetailReciprocalInsuredTimeGT30', 'LEDGER', NULL, '2.01', 'Retail', 'Insured', NULL),
    ('deposits_reg', 'RetailReciprocalInsuredTimeLT30', 'LEDGER', NULL, '2.02', 'Retail', 'Insured', NULL),
    ('deposits_reg', 'RetailReciprocalUninsuredNMD', 'LEDGER', NULL, '2.03', 'Retail', 'Uninsured', 'Moved from Other Retail per prior workbook (32.g.6)'),
    ('deposits_reg', 'RetailReciprocalUninsuredTimeGT30', 'LEDGER', NULL, '2.03', 'Retail', 'Uninsured', 'Moved from Other Retail per prior workbook (32.g.6)'),
    ('deposits_reg', 'RetailReciprocalUninsuredTimeLT30', 'LEDGER', NULL, '2.04', 'Retail', 'Uninsured', 'Moved from Other Retail per prior workbook (32.g.6)'),
    ('deposits_reg', 'RetailTraditionalInsuredNMD', 'LEDGER', NULL, '1.01', 'Retail', 'Insured', NULL),
    ('deposits_reg', 'RetailTraditionalInsuredTimeGT30', 'LEDGER', NULL, '1.01', 'Retail', 'Insured', NULL),
    ('deposits_reg', 'RetailTraditionalInsuredTimeLT30', 'LEDGER', NULL, '1.02', 'Retail', 'Insured', NULL),
    ('deposits_reg', 'RetailTraditionalUninsuredNMD', 'LEDGER', NULL, '1.03', 'Retail', 'Uninsured', NULL),
    ('deposits_reg', 'RetailTraditionalUninsuredTimeGT30', 'LEDGER', NULL, '1.03', 'Retail', 'Uninsured', NULL),
    ('deposits_reg', 'RetailTraditionalUninsuredTimeLT30', 'LEDGER', NULL, '1.04', 'Retail', 'Uninsured', NULL),
    ('deposits_reg', 'SecuredPSENMD', 'LEDGER', NULL, '8.01', 'Secured', 'n/a', 'Feeds Outflow - Secured (balance only here)'),
    ('deposits_reg', 'SecuredPSETimeLT30', 'LEDGER', NULL, '8.01', 'Secured', 'n/a', 'Feeds Outflow - Secured (balance only here)'),
    ('deposits_reg', 'SecuredPSETimeGT30', 'LEDGER', NULL, '8.01', 'Secured', 'n/a', 'Feeds Outflow - Secured (balance only here)'),
    ('deposits_reg', 'SecuredNonPSENMD', 'LEDGER', NULL, '8.02', 'Secured', 'n/a', 'Feeds Outflow - Secured (balance only here)'),
    ('deposits_reg', 'SecuredNonPSETimeLT30', 'LEDGER', NULL, '8.02', 'Secured', 'n/a', 'Feeds Outflow - Secured (balance only here)'),
    ('deposits_reg', 'SecuredNonPSETimeGT30', 'LEDGER', NULL, '8.02', 'Secured', 'n/a', 'Feeds Outflow - Secured (balance only here)'),
    ('deposits_reg', 'SecuredRecipDepositsPSE', 'LEDGER', NULL, '3.08', 'NFSE', 'n/a', 'Secured reciprocal, PSE collateral (2026-11-17, per user: 40%, line 3.08)'),
    ('deposits_reg', 'SecuredRecipDepositsNonPSE', 'LEDGER', NULL, '3.08', 'NFSE', 'n/a', 'Secured reciprocal, non-PSE collateral (2026-11-17, per user: 40%, line 3.08)'),
    ('deposits_reg', 'WholesaleReciprocalInsuredNMD', 'LEDGER', NULL, '3.08', 'NFSE', 'Insured', NULL),
    ('deposits_reg', 'WholesaleReciprocalInsuredTimeGT30', 'LEDGER', NULL, '3.10', 'NFSE', 'Insured', NULL),
    ('deposits_reg', 'WholesaleReciprocalInsuredTimeLT30', 'LEDGER', NULL, '3.09', 'NFSE', 'Insured', NULL),
    ('deposits_reg', 'WholesaleReciprocalUninsuredNMD', 'LEDGER', NULL, '3.08', 'NFSE', 'Uninsured', NULL),
    ('deposits_reg', 'WholesaleReciprocalUninsuredTimeGT30', 'LEDGER', NULL, '3.10', 'NFSE', 'Uninsured', NULL),
    ('deposits_reg', 'WholesaleReciprocalUninsuredTimeLT30', 'LEDGER', NULL, '3.09', 'NFSE', 'Uninsured', NULL),
    ('deposits_reg', 'WholesaleTraditionalInsuredNMD', 'LEDGER', NULL, '3.03', 'NFSE', 'Insured', NULL),
    ('deposits_reg', 'WholesaleTraditionalInsuredTimeGT30', 'LEDGER', NULL, '3.06', 'NFSE', 'Insured', NULL),
    ('deposits_reg', 'WholesaleTraditionalInsuredTimeLT30', 'LEDGER', NULL, '3.05', 'NFSE', 'Insured', NULL),
    ('deposits_reg', 'WholesaleTraditionalUninsuredNMD', 'LEDGER', NULL, '3.04', 'NFSE', 'Uninsured', NULL),
    ('deposits_reg', 'WholesaleTraditionalUninsuredTimeGT30', 'LEDGER', NULL, '3.07', 'NFSE', 'Uninsured', NULL),
    ('deposits_reg', 'WholesaleTraditionalUninsuredTimeLT30', 'LEDGER', NULL, '3.05', 'NFSE', 'Uninsured', NULL),
    ('wholesalefunding', 'BrokeredNMD', 'LEDGER', NULL, '2.08', 'Brokered', 'n/a', 'Brokered NMD, any maturity bucket'),
    ('wholesalefunding', 'BrokeredTermNMDLT30', 'LEDGER', NULL, '2.06', 'Brokered', 'n/a', 'Term NMD maturing <=30d, treated like brokered CDs <=30d (100%).'),
    ('wholesalefunding', 'BrokeredTermNMDGT30', 'LEDGER', NULL, '2.09', 'Brokered', 'n/a', NULL),
    ('wholesalefunding', 'BrokeredCDLT30', 'LEDGER', NULL, '2.06', 'Brokered', 'n/a', NULL),
    ('wholesalefunding', 'BrokeredCDGT30', 'LEDGER', NULL, '2.07', 'Brokered', 'n/a', NULL),
    ('wholesalefunding', 'FHLBAdvanceLT30', 'LEDGER', NULL, '4.05', 'FHLB', 'n/a', NULL),
    ('wholesalefunding', 'FHLBAdvanceGT30', 'LEDGER', NULL, '4.06', 'FHLB', 'n/a', 'Memo line: outside the 30 day window'),
    ('wholesalefunding', 'FedFundsPurchasedLT30', 'LEDGER', NULL, '5.01', 'FedFunds', 'n/a', NULL),
    ('wholesalefunding', 'FedFundsPurchasedGT30', 'LEDGER', NULL, '5.06', 'FedFunds', 'n/a', 'Memo line: outside the 30 day window'),
    ('wholesalefunding', 'LTDSeniorLT30', 'LEDGER', NULL, '5.02', 'LTD', 'n/a', NULL),
    ('wholesalefunding', 'LTDSeniorGT30', 'LEDGER', NULL, '5.04', 'LTD', 'n/a', 'Memo line: outside the 30 day window'),
    ('wholesalefunding', 'LTDSubordinatedLT30', 'LEDGER', NULL, '5.02', 'LTD', 'n/a', NULL),
    ('wholesalefunding', 'LTDSubordinatedGT30', 'LEDGER', NULL, '5.04', 'LTD', 'n/a', 'Memo line: outside the 30 day window'),
    ('wholesalefunding', 'LTDTrustPreferredLT30', 'LEDGER', NULL, '5.02', 'LTD', 'n/a', NULL),
    ('wholesalefunding', 'LTDTrustPreferredGT30', 'LEDGER', NULL, '5.04', 'LTD', 'n/a', 'Memo line: outside the 30 day window'),
    ('wholesalefunding', 'LTDOtherLT30', 'LEDGER', NULL, '5.02', 'LTD', 'n/a', 'Long-Term Debt not senior, subordinated or trust preferred'),
    ('wholesalefunding', 'LTDOtherGT30', 'LEDGER', NULL, '5.04', 'LTD', 'n/a', 'Memo line: outside the 30 day window'),
    ('wholesalefunding', 'WFUnclassifiedLT30', 'LEDGER', NULL, '9.99', 'Review', 'n/a', 'Instrument type not recognised. Add a CASE branch in lcr_wholesale_funding.'),
    ('wholesalefunding', 'WFUnclassifiedGT30', 'LEDGER', NULL, '9.99', 'Review', 'n/a', 'Instrument type not recognised. Add a CASE branch in lcr_wholesale_funding.')
  AS t(source_system, lcr_class, rule, line_number_op, line_number, entity_type, insurance_status, note);

SELECT source_system, rule, COUNT(*) AS classes FROM finance.liquidity.lcr_class_map GROUP BY ALL ORDER BY 1, 2;

In [0]:
%sql
WITH ld AS (SELECT * FROM finance.liquidity.lcr_line_def),
cm AS (SELECT * FROM finance.liquidity.lcr_class_map),
rf AS (SELECT * FROM finance.liquidity.lcr_runoff_factors),
checks AS (
  SELECT 1 AS check_order, 'Duplicate line_number' AS check_name, COUNT(1) - COUNT(DISTINCT line_number) AS issues FROM ld
  UNION ALL
  SELECT 2, 'Duplicate source_system + lcr_class', COUNT(1) - COUNT(DISTINCT source_system, lcr_class) FROM cm
  UNION ALL
  SELECT 3, 'Map points to a line not in lcr_line_def', COUNT_IF(ld.line_number IS NULL)
  FROM cm LEFT JOIN ld ON ld.line_number = cm.line_number
  UNION ALL
  SELECT 4, 'SPLIT op line missing or not in lcr_line_def', COUNT_IF(ld.line_number IS NULL)
  FROM cm LEFT JOIN ld ON ld.line_number = cm.line_number_op WHERE cm.rule = 'SPLIT'
  UNION ALL
  SELECT 5, 'Deposit runoff class with no line map', COUNT(1)
  FROM rf LEFT JOIN cm ON cm.lcr_class = rf.lcr_class AND cm.source_system = 'deposits_reg'
  WHERE cm.lcr_class IS NULL AND rf.lcr_class NOT LIKE '%ExcessOperational%' AND rf.lcr_class <> 'BrokeredNMDGT30'
)
SELECT check_name, issues, CASE WHEN issues = 0 THEN 'PASS' ELSE 'FAIL' END AS status
FROM checks ORDER BY check_order;

In [0]:
%sql
CREATE OR REPLACE VIEW finance.liquidity.v_lcr_assumptions_current AS
SELECT
framework, assumption_category, assumption_code, assumption_name,
assumption_value, assumption_description, effective_date, owner, modified_by, modified_ts
FROM finance.liquidity.assumption_reference
WHERE framework = 'LCR'
AND active_flag = 'Y'
AND effective_date <= current_date()
AND (expiration_date IS NULL OR expiration_date >= current_date());
SELECT
assumption_code, assumption_name, assumption_value, effective_date,
expiration_date, active_flag, comments, modified_by, modified_ts
FROM finance.liquidity.assumption_reference
WHERE framework = 'LCR'
ORDER BY assumption_code, effective_date DESC;
